# V14 GlobalNN — Kaggle T4×2 / P100 fine-tune

Transformer-encoder with learned categorical embeddings for Партнер/Артикул/Бренд/Канал. 192-dim, 4 layers, 8 heads, 5-quantile pinball head. Trains in one Kaggle session (~90 min on T4 ×2).

**Output:** `preds_v14_globalnn_{val,test}.csv` to `/kaggle/working/`. Pull via `kaggle kernels output mykhailokozyrev/bpm-v14-globalnn`.

In [ ]:
# Cell 1: GPU + PyTorch compatibility shim
# Kaggle's preinstalled PyTorch 2.10 only supports sm_70+ GPUs.
# If we got assigned a P100 (sm_60), we MUST reinstall PyTorch
# to a version that still supports it (2.5.x or earlier).
import subprocess, sys
import torch

def gpu_capability_supported():
    if not torch.cuda.is_available():
        return False
    cap = torch.cuda.get_device_capability(0)
    cap_int = cap[0] * 10 + cap[1]  # 60 for P100, 75 for T4, 80 for A100
    # Quick test: try a tiny CUDA op. If it crashes, capability mismatch.
    try:
        x = torch.zeros(2, device='cuda'); x + 1
        return True
    except Exception:
        return False

if torch.cuda.is_available() and not gpu_capability_supported():
    cap = torch.cuda.get_device_capability(0)
    name = torch.cuda.get_device_name(0)
    print(f'GPU {name} (sm_{cap[0]}{cap[1]}) is incompatible with torch {torch.__version__}.')
    print('Reinstalling torch 2.5.1+cu121 (supports sm_60 through sm_90)...')
    subprocess.check_call([
        sys.executable, '-m', 'pip', 'install', '--quiet',
        '--index-url', 'https://download.pytorch.org/whl/cu121',
        'torch==2.5.1', 'torchvision==0.20.1',
    ])
    print('Re-importing torch...')
    # Need to fully reload torch
    for mod in list(sys.modules):
        if mod.startswith('torch'):
            del sys.modules[mod]
    import torch

import numpy as np, pandas as pd, json, os
print('torch', torch.__version__, 'cuda', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')
if torch.cuda.is_available():
    cap = torch.cuda.get_device_capability(0)
    print(f'GPU capability: sm_{cap[0]}{cap[1]}')
    # Final smoke test
    x = torch.zeros(2, device='cuda'); y = x + 1
    print(f'CUDA smoke test OK: {y}')
assert torch.cuda.is_available(), 'GPU required'

In [ ]:
# Cell 2: locate the data
DATA_DIR = '/kaggle/input/bpm-v14-globalnn-data'
OUT_DIR  = '/kaggle/working'
for f in ('train.parquet', 'val.parquet', 'test.parquet',
          'vocab.json', 'manifest.json'):
    assert os.path.exists(f'{DATA_DIR}/{f}'), f'missing {f}'

with open(f'{DATA_DIR}/manifest.json') as fh:
    manifest = json.load(fh)
with open(f'{DATA_DIR}/vocab.json') as fh:
    vocab = json.load(fh)
print('manifest:', manifest['n_partners'], 'partners,',
      manifest['n_skus'], 'skus,',
      manifest['n_brands'], 'brands,',
      manifest['n_channels'], 'channels')

In [ ]:
# Cell 3: load + scrub + tensorise
train_df = pd.read_parquet(f'{DATA_DIR}/train.parquet')
val_df   = pd.read_parquet(f'{DATA_DIR}/val.parquet')
test_df  = pd.read_parquet(f'{DATA_DIR}/test.parquet')
print('loaded:', train_df.shape, val_df.shape, test_df.shape)

CAT_COLS = ['Партнер_idx', 'Артикул_idx', 'Бренд_idx', 'Канал_idx']
DROP_COLS = ['Период_str', 'target_qty'] + CAT_COLS
NUMERIC_COLS = [c for c in train_df.columns if c not in DROP_COLS
                 and pd.api.types.is_numeric_dtype(train_df[c])]
print('n_numeric:', len(NUMERIC_COLS))

# Robust scrub: catch BOTH NaN and Inf (upstream feature engineering
# can produce Inf via division-by-zero in ratio columns)
for df in (train_df, val_df, test_df):
    arr = df[NUMERIC_COLS].values.astype(np.float32)
    arr = np.nan_to_num(arr, nan=0.0, posinf=1e6, neginf=-1e6)
    df[NUMERIC_COLS] = arr
for split_name, df in [('train', train_df), ('val', val_df), ('test', test_df)]:
    bad = df['target_qty'].isna() | np.isinf(df['target_qty'])
    if bad.any():
        df.drop(df.index[bad], inplace=True)
        print(f'  [scrub] {split_name}: dropped {bad.sum()} bad target rows')

mu = train_df[NUMERIC_COLS].mean().values.astype(np.float32)
sd = train_df[NUMERIC_COLS].std().fillna(1.0).replace(0, 1).values.astype(np.float32)
mu = np.nan_to_num(mu, nan=0.0, posinf=0.0, neginf=0.0)
sd = np.nan_to_num(sd, nan=1.0, posinf=1.0, neginf=1.0)
print(f'  mu range [{mu.min():.3g}, {mu.max():.3g}]  sd range [{sd.min():.3g}, {sd.max():.3g}]')

def make_tensors(df):
    p = torch.tensor(df['Партнер_idx'].values, dtype=torch.long)
    s = torch.tensor(df['Артикул_idx'].values, dtype=torch.long)
    b = torch.tensor(df['Бренд_idx'].values,   dtype=torch.long)
    c = torch.tensor(df['Канал_idx'].values,   dtype=torch.long)
    n = (df[NUMERIC_COLS].values.astype(np.float32) - mu) / sd
    n = np.nan_to_num(n, nan=0.0, posinf=10.0, neginf=-10.0)
    n = np.clip(n, -10.0, 10.0)
    n = torch.tensor(n, dtype=torch.float32)
    y = torch.tensor(df['target_qty'].values.astype(np.float32), dtype=torch.float32)
    return p, s, b, c, n, y

train_tensors = make_tensors(train_df)
val_tensors   = make_tensors(val_df)
test_tensors  = make_tensors(test_df)
for name, t in [('train_n', train_tensors[4]), ('train_y', train_tensors[5]),
                ('val_n', val_tensors[4]),     ('test_n', test_tensors[4])]:
    assert torch.isfinite(t).all(), f'NaN/Inf in {name}!'
print('✓ all tensors finite')

In [ ]:
# Cell 4: GlobalNN architecture (192-dim Transformer-encoder)
import torch.nn as nn
from dataclasses import dataclass

@dataclass
class Cfg:
    n_partners: int; n_skus: int; n_brands: int; n_channels: int; n_numeric: int
    emb_dim: int = 32
    num_enc_dim: int = 64
    d_model: int = 192
    nhead: int = 8
    n_layers: int = 4
    dropout: float = 0.10
    quantiles: tuple = (0.1, 0.25, 0.5, 0.75, 0.9)

class GlobalNN(nn.Module):
    def __init__(self, cfg):
        super().__init__(); self.cfg = cfg
        self.emb_partner = nn.Embedding(cfg.n_partners, cfg.emb_dim)
        self.emb_sku     = nn.Embedding(cfg.n_skus,     cfg.emb_dim)
        self.emb_brand   = nn.Embedding(cfg.n_brands,   cfg.emb_dim)
        self.emb_channel = nn.Embedding(cfg.n_channels, cfg.emb_dim)
        self.num_enc = nn.Sequential(
            nn.Linear(cfg.n_numeric, cfg.num_enc_dim),
            nn.GELU(), nn.Dropout(cfg.dropout),
            nn.Linear(cfg.num_enc_dim, cfg.num_enc_dim))
        assert (4*cfg.emb_dim + cfg.num_enc_dim) == cfg.d_model
        enc_layer = nn.TransformerEncoderLayer(
            d_model=cfg.d_model, nhead=cfg.nhead,
            dim_feedforward=cfg.d_model*4,
            dropout=cfg.dropout, batch_first=True)
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=cfg.n_layers)
        self.q_head = nn.Linear(cfg.d_model, len(cfg.quantiles))
    def forward(self, p, s, b, c, n):
        x = torch.cat([self.emb_partner(p), self.emb_sku(s),
                       self.emb_brand(b), self.emb_channel(c),
                       self.num_enc(n)], dim=-1).unsqueeze(1)
        return self.q_head(self.encoder(x).squeeze(1))

def pinball_loss(y_pred, y_true, quantiles):
    losses = []
    for i, q in enumerate(quantiles):
        diff = y_true - y_pred[:, i]
        losses.append(torch.maximum(q*diff, (q-1)*diff))
    return torch.stack(losses, dim=-1).mean()

cfg = Cfg(n_partners=manifest['n_partners'], n_skus=manifest['n_skus'],
          n_brands=manifest['n_brands'], n_channels=manifest['n_channels'],
          n_numeric=len(NUMERIC_COLS))
model = GlobalNN(cfg).cuda()
n_params = sum(p.numel() for p in model.parameters())
print(f'GlobalNN: {n_params/1e6:.2f}M params')

In [ ]:
# Cell 5: train (~90-120 min on T4)
from torch.utils.data import DataLoader, TensorDataset
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

BATCH = 1024; EPOCHS = 15; LR = 5e-4; WD = 0.01

train_ds = TensorDataset(*train_tensors)
val_ds   = TensorDataset(*val_tensors)
train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True,
                          num_workers=2, drop_last=True, pin_memory=True)
val_loader   = DataLoader(val_ds, batch_size=BATCH*4, shuffle=False,
                          num_workers=2, pin_memory=True)

opt = AdamW(model.parameters(), lr=LR, weight_decay=WD)
sched = CosineAnnealingLR(opt, T_max=EPOCHS*len(train_loader))

# Smoke test BEFORE the loop
model.train()
smoke = [t[:64].cuda() for t in train_tensors]
sm_pred = model(smoke[0], smoke[1], smoke[2], smoke[3], smoke[4])
sm_loss = pinball_loss(sm_pred, smoke[5], cfg.quantiles)
print(f'smoke: loss={sm_loss.item():.4f}  finite={torch.isfinite(sm_loss).item()}')
assert torch.isfinite(sm_loss).item(), 'non-finite loss — abort'
print('✓ starting training')

best_val = float('inf')
ckpt = '/kaggle/working/v14_globalnn_best.pt'

for epoch in range(EPOCHS):
    model.train()
    epoch_loss = 0.0
    for batch in train_loader:
        p, s, b, c, n, y = [t.cuda(non_blocking=True) for t in batch]
        opt.zero_grad()
        loss = pinball_loss(model(p, s, b, c, n), y, cfg.quantiles)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step(); sched.step()
        epoch_loss += loss.item()
    model.eval()
    with torch.no_grad():
        v_loss = 0.0
        for batch in val_loader:
            p, s, b, c, n, y = [t.cuda(non_blocking=True) for t in batch]
            v_loss += pinball_loss(model(p, s, b, c, n), y, cfg.quantiles).item()
        v_loss /= len(val_loader)
    flag = '★' if v_loss < best_val else ' '
    if v_loss < best_val:
        best_val = v_loss
        torch.save({'sd': model.state_dict(), 'mu': mu, 'sd_arr': sd,
                    'numeric_cols': NUMERIC_COLS, 'cfg': cfg.__dict__},
                   ckpt)
    print(f'epoch {epoch+1:2d}/{EPOCHS}  '
          f'train={epoch_loss/len(train_loader):.4f}  val={v_loss:.4f}  {flag}')
print(f'\nbest val loss = {best_val:.4f}; saved to {ckpt}')

In [ ]:
# Cell 6: forecast on val + test (~5 min)
ck = torch.load(ckpt, map_location='cuda', weights_only=False)
model.load_state_dict(ck['sd']); model.eval()

QUANTILES = list(cfg.quantiles)
MEDIAN_IDX = QUANTILES.index(0.5)
inv_partner = {v: k for k, v in vocab['Партнер'].items()}
inv_sku     = {v: k for k, v in vocab['Артикул'].items()}

def predict(tensors, df):
    p, s, b, c, n, y = [t.cuda() for t in tensors]
    preds = []
    with torch.no_grad():
        for i in range(0, len(p), BATCH*4):
            sl = slice(i, i+BATCH*4)
            preds.append(model(p[sl], s[sl], b[sl], c[sl], n[sl]).cpu().float().numpy())
    pred_all = np.concatenate(preds, axis=0)
    out = df[['Период_str']].rename(columns={'Период_str': 'Период'}).copy()
    out['Партнер'] = df['Партнер_idx'].map(inv_partner).fillna('<UNK>')
    out['Артикул'] = df['Артикул_idx'].map(inv_sku).fillna('<UNK>')
    out['target_qty'] = df['target_qty'].values
    out['prediction'] = np.clip(pred_all[:, MEDIAN_IDX], 0, None)
    return out

val_pred  = predict(val_tensors,  val_df)
test_pred = predict(test_tensors, test_df)
val_pred.to_csv('/kaggle/working/preds_v14_globalnn_val.csv', index=False)
test_pred.to_csv('/kaggle/working/preds_v14_globalnn_test.csv', index=False)
print(f'wrote val ({len(val_pred):,}), test ({len(test_pred):,})')

for name, df in (('val', val_pred), ('test', test_pred)):
    y, yh = df['target_qty'].values.astype(float), df['prediction'].values.astype(float)
    wape = np.abs(y-yh).sum() / max(1e-9, y.sum())
    bias = (yh.sum()-y.sum()) / max(1e-9, y.sum()) * 100
    print(f'{name}: rows={len(df):,}  WAPE={wape:.3f}  bias%={bias:+.1f}')